In [8]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

#### Summarization Middleware in LangChain is used to automatically compress long conversations when the context becomes too large.

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

main_model = ChatGroq(
    model="qwen/qwen3-32b"
)

summary_model = ChatGroq(
    model="llama-3.1-8b-instant"
)

agent = create_agent(
    model=main_model,
    middleware=[
        SummarizationMiddleware(
            model=summary_model,
            max_tokens_before_summary=40,
            messages_to_keep=10,
        )
    ],
    checkpointer=checkpointer,
)

C:\Users\Pratik Giri\AppData\Local\Temp\ipykernel_17956\122688434.py:19: DeprecationWarning: max_tokens_before_summary is deprecated. Use trigger=('tokens', value) instead.
  SummarizationMiddleware(
C:\Users\Pratik Giri\AppData\Local\Temp\ipykernel_17956\122688434.py:19: DeprecationWarning: messages_to_keep is deprecated. Use keep=('messages', value) instead.
  SummarizationMiddleware(


In [11]:
## Run with thread 
# In LangGraph/LangChain agents, a thread is used to maintain conversation state across multiple runs.

# Without a thread, every call is independent.
config = {
    "configurable": {
        "thread_id": "user_123"
    }
}

In [15]:
response=agent.invoke(
    {"messages": [{"role": "user", "content": "My name is Pratik"}]},
    config=config
)
print(response)

response1=agent.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config=config
)
print("----------------------\n",response)

{'messages': [HumanMessage(content='My name is Pratik', additional_kwargs={}, response_metadata={}, id='3103b852-23f4-4171-906d-f2f93fa7607d'), AIMessage(content='<think>\nOkay, the user introduced themselves as Pratik. Let me see how to respond. First, I should acknowledge their name and maybe ask how I can assist them. I need to keep it friendly and open-ended. Let me check if there\'s any previous context, but since this is the start, just a simple greeting should work. Make sure to use proper grammar and a welcoming tone. Alright, something like, "Hello, Pratik! How can I assist you today?" That should do it.\n</think>\n\nHello, Pratik! How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 121, 'prompt_tokens': 14, 'total_tokens': 135, 'completion_time': 0.255570861, 'completion_tokens_details': None, 'prompt_time': 0.001116703, 'prompt_tokens_details': None, 'queue_time': 3.613952747, 'total_time': 0.256687564}, 'model_name': '

InternalServerError: Error code: 503 - {'error': {'message': 'qwen/qwen3-32b is currently over capacity. Please try again and back off exponentially. Visit https://groqstatus.com to see if there is an active incident.', 'type': 'internal_server_error'}}


#### Human-in-the-loop
Human-in-the-Loop (HITL) is a mechanism where an AI agent pauses execution at predefined decision points and requires human review or approval before proceeding, improving safety, reliability, and control over autonomous actions. In LangGraph, HITL is typically implemented using interrupts and resumable graph execution backed by a checkpointer.

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# --------------------------------------------------
# Tools
# --------------------------------------------------

@tool
def read_email(email_id: str) -> str:
    """Read an email by ID."""
    return f"""
From: boss@company.com
Subject: Project Update

Hello,
Please send me the project status by tomorrow.
"""


@tool
def send_email(
    recipient: str,
    subject: str,
    body: str
) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {recipient} with subject '{subject}'"


# --------------------------------------------------
# Model
# --------------------------------------------------

model = ChatGroq(
    model="llama-3.1-8b-instant"
)

# --------------------------------------------------
# Agent
# --------------------------------------------------

agent = create_agent(
    model=model,
    tools=[read_email, send_email],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": {
                    "allowed_decisions": [
                        "approve",
                        "reject",
                        "edit"
                    ]
                },
                "read_email": False,
            }
        )
    ],
)

# --------------------------------------------------
# Thread Config
# --------------------------------------------------

config = {
    "configurable": {
        "thread_id": "user_456"
    }
}

# --------------------------------------------------
# Example 1
# No interruption
# --------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Read email with ID 12345"
            }
        ]
    },
    config=config
)

print("\n=== READ EMAIL RESPONSE ===")
print(response)

# --------------------------------------------------
# Example 2
# This WILL trigger Human Approval
# --------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """
                Send an email to john@example.com.

                Subject: Meeting

                Body:
                Let's meet tomorrow at 10 AM.
                """
            }
        ]
    },
    config=config
)

print("\n=== INTERRUPT RESPONSE ===")
print(response)

# --------------------------------------------------
# Check if approval needed
# --------------------------------------------------

if "__interrupt__" in response:

    print("\nApproval Required!")

    # Approve the action

    response = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "approve"
                    }
                ]
            }
        ),
        config=config
    )

    print("\n=== FINAL RESPONSE ===")
    print(response)